# Đánh giá agent: ReAct tự viết, native tool calling, JSON schema

Một agent "chạy được" trên ba câu thử chưa nói lên gì. Bài này đo agent như đo một model: **bộ task ghim** có đáp án tính
từ dữ liệu (24 câu hỏi về một hiệu sách tự tạo, `bookstore.py`), chấm tự động, khoảng tin cậy cho tỉ lệ thành công, so
theo cặp giữa hai cách, và **phân loại lỗi** tự động để biết agent hỏng ở đâu. Ba cách gọi tool trên cùng `qwen2.5:3b`:

- **ReAct văn bản**: vòng `run_react` của bạn (bài Basic 02), model viết `Thought / Action / Action Input`.
- **Native**: trường `tools` của Ollama, chat template của Qwen2.5 lo định dạng (module 11 mục 6).
- **ReAct JSON**: cùng vòng suy nghĩ nhưng mỗi lượt ép output bằng JSON schema (`format` của Ollama).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b`. Các cell `explore` gọi model khoảng 250 lượt.
- Chấm: `uv run pytest tracks/ai-engineer/modules/16-ai-agents/exercises/intermediate/02-agent-eval -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import math
import re
import time
import unicodedata
from dataclasses import asdict

import pandas as pd

import bookstore

## 1. Dán agent của bạn

Notebook tự chứa: dán lời giải bài Basic 01 (`ToolRegistry` và các hàm đi kèm) và bài Basic 02 (`parse_react`,
`run_react` và các dataclass) vào cell dưới. Test của bài này không chấm cell đó, nhưng mọi cell `explore` dùng nó.

In [ ]:
# TODO: dán nội dung tools.py (bài Basic 01) rồi react.py (bài Basic 02) vào đây, bỏ các dòng import trùng nếu muốn.

## 2. Chấm câu trả lời

Câu trả lời của agent là chữ tự do: "Tổng là 165.000 đồng", "khoảng 5,31 USD", "2,3 triệu". `numbers_in(text)` trả mọi
con số trong chuỗi theo thứ tự xuất hiện, kiểu `float`, theo quy ước viết số tiếng Việt:

- Nhóm 3 chữ số cách nhau bởi `.`, `,` hoặc dấu cách sau 1 tới 3 chữ số đầu là **phân cách hàng nghìn**:
  `1.234.567`, `165,000`, `1 000 000`.
- Một dấu `.` hoặc `,` không theo dạng trên là **dấu thập phân**: `5,31`, `2.84`, `0.8`.
- Số theo sau bởi ` triệu` nhân $$10^6$$, bởi ` nghìn` hoặc ` ngàn` nhân $$10^3$$ (có hoặc không có dấu cách).
- Không đọc số âm; chữ số dính chữ cái vẫn tính (`B19` cho 19).

`check_answer(answer, expected)`: `answer` rỗng hoặc `None` thì `False`. `expected` là list: phần tử chuỗi phải xuất
hiện trong câu trả lời (không phân biệt hoa thường, sau chuẩn hoá NFC); phần tử số phải khớp **một** số trong
`numbers_in(answer)` với sai lệch tuyệt đối không quá `0.01`. Mọi phần tử phải đạt.

In [ ]:
def numbers_in(text: str) -> list[float]:
    """Mọi con số trong text theo thứ tự xuất hiện (quy ước ở trên)."""
    # TODO
    raise NotImplementedError


def check_answer(answer: str | None, expected: list) -> bool:
    """Mọi phần tử của expected đều có trong answer (chuỗi: chứa, không phân biệt hoa thường; số: sai lệch <= 0.01)."""
    # TODO
    raise NotImplementedError

## 3. Tỉ lệ thành công có khoảng tin cậy, so theo cặp, phân loại lỗi

`wilson_interval(k, n, z=1.96)`: khoảng tin cậy Wilson cho tỉ lệ $$\hat p = k/n$$ (mục 2 lý thuyết), trả `(lo, hi)`;
`n = 0` thì `(0.0, 1.0)`.

$$
\frac{\hat p + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}
$$

`discordant(a, b)`: `a`, `b` là dict `task_id -> bool` của hai cách trên **cùng** bộ task (khoá khác nhau thì
`ValueError`). Trả `(chỉ_a_đúng, chỉ_b_đúng)`, mỗi phần là list id đã sắp xếp. Task cả hai cùng đúng hoặc cùng sai
không nói gì về chênh lệch; chỉ các cặp lệch nhau mới nói (ý của kiểm định McNemar).

`classify_failure(row)`: `row` là dict có `ok`, `stopped`, `answer`, `question`, `steps` (list dict có `kind`,
`observation`, `ok`, như `asdict(Step)`). Trả nhãn đầu tiên khớp theo thứ tự:

1. `"ok"`: `row["ok"]` đúng.
2. `"no_answer"`: `stopped` khác `"answer"` (hết bước, lỗi định dạng, lặp).
3. `"no_tool"`: không có bước nào `kind == "action"`.
4. `"hallucinated_number"`: câu trả lời có một con số (theo `numbers_in`) không khớp (sai lệch quá `0.01`) với số nào
   trong các observation hay trong câu hỏi: agent viết ra số không lấy từ đâu cả.
5. `"tool_error"`: có bước action với `ok` sai.
6. `"wrong_other"`: còn lại.

In [ ]:
def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    # TODO
    raise NotImplementedError


def discordant(a: dict[str, bool], b: dict[str, bool]) -> tuple[list[str], list[str]]:
    # TODO
    raise NotImplementedError


def classify_failure(row: dict) -> str:
    # TODO
    raise NotImplementedError

## 4. Chạy thật trên 24 task

Ba cách dùng chung `ToolRegistry` với 5 tool của `bookstore.py`, `temperature` 0, `seed` 0, tối đa 8 bước. Client
nhỏ dưới đây gọi thẳng `/api/chat` của Ollama (giống `llmkit` nhưng có thêm `seed`, `tools` và đếm token), kèm vòng
native và vòng ReAct JSON (đây là phần harness, không phải bài tập). Cả ba cách chạy khoảng 10 tới 15 phút trên PC của
track; cách JSON chậm nhất.

In [ ]:
# explore: client Ollama có seed, vòng native và vòng ReAct JSON
import httpx

import llmkit

OLLAMA = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MODEL = "qwen2.5:3b"


class OllamaChat:
    def __init__(self, model=MODEL, seed=0, num_ctx=None):
        self.model, self.seed, self.num_ctx = model, seed, num_ctx
        self.http = httpx.Client(timeout=600)
        self.prompt_tokens = []

    def raw(self, messages, tools=None, json_schema=None):
        options = {"temperature": 0, "seed": self.seed}
        if self.num_ctx:
            options["num_ctx"] = self.num_ctx
        payload = {"model": self.model, "messages": messages, "stream": False, "options": options}
        if tools:
            payload["tools"] = tools
        if json_schema is not None:
            payload["format"] = json_schema
        r = self.http.post(OLLAMA + "/api/chat", json=payload)
        if r.status_code >= 400:
            raise llmkit.LLMError(f"HTTP {r.status_code}: {r.text[:200]}")
        data = r.json()
        self.prompt_tokens.append(data.get("prompt_eval_count") or 0)
        return data["message"]

    def chat(self, messages, *, model=None, temperature=0.0, json_schema=None):
        return self.raw(messages, json_schema=json_schema)["content"]


NATIVE_SYSTEM = ("Bạn là trợ lý của một hiệu sách. Dùng tool khi cần số liệu; số liệu phải lấy từ kết quả tool. "
                 "Trả lời tiếng Việt, ngắn gọn.")


def run_native(client, task, registry, max_steps=8):
    messages = [{"role": "system", "content": NATIVE_SYSTEM}, {"role": "user", "content": task}]
    result = RunResult(None, "max_steps", [], messages)
    for index in range(max_steps):
        start = time.perf_counter()
        msg = client.raw(messages, tools=registry.specs())
        calls = msg.get("tool_calls") or []
        if not calls:
            result.answer, result.stopped = msg.get("content", ""), "answer"
            result.steps.append(Step(index, "final", msg.get("content", ""), seconds=time.perf_counter() - start))
            return result
        messages.append({"role": "assistant", "content": msg.get("content", ""),
                         "tool_calls": [{"function": c["function"]} for c in calls]})
        for c in calls:
            name, args = c["function"]["name"], c["function"].get("arguments", {})
            obs = registry.call(name, args)
            result.tool_errors += not obs.ok
            messages.append({"role": "tool", "content": obs.content, "tool_name": name})
            result.steps.append(Step(index, "action", json.dumps(c["function"], ensure_ascii=False), name,
                                     args if isinstance(args, dict) else None, obs.content, obs.ok,
                                     time.perf_counter() - start))
    return result


JSON_SYSTEM = SYSTEM_TEMPLATE.split("Mỗi lượt")[0] + (
    'Mỗi lượt, trả lời bằng MỘT JSON object:\n'
    '- cần gọi tool: {{"thought": "...", "action": "<tên tool>", "action_input": {{...}}}}\n'
    '- đã đủ thông tin: {{"thought": "...", "action": "Final Answer", "final_answer": "<câu trả lời tiếng Việt>"}}\n'
    "Sau mỗi action, hệ thống trả về `Observation: <kết quả>`. Số liệu phải lấy từ Observation, không tự bịa.")


def run_react_json(client, task, registry, max_steps=8):
    names = [s["function"]["name"] for s in registry.specs()] + ["Final Answer"]
    schema = {"type": "object", "required": ["thought", "action"], "properties": {
        "thought": {"type": "string"}, "action": {"type": "string", "enum": names},
        "action_input": {"type": "object"}, "final_answer": {"type": "string"}}}
    messages = [{"role": "system", "content": JSON_SYSTEM.format(tools=render_tools(registry.specs()))},
                {"role": "user", "content": task}]
    result = RunResult(None, "max_steps", [], messages)
    for index in range(max_steps):
        start = time.perf_counter()
        reply = client.chat(messages, json_schema=schema)
        messages.append({"role": "assistant", "content": reply})
        try:
            data = json.loads(reply)
            action = data["action"]
        except (ValueError, KeyError, TypeError):
            result.format_errors += 1
            messages.append({"role": "user", "content": "Lỗi định dạng: cần JSON object có thought, action."})
            result.steps.append(Step(index, "format_error", reply, seconds=time.perf_counter() - start))
            continue
        if action == "Final Answer":
            result.answer, result.stopped = data.get("final_answer", ""), "answer"
            result.steps.append(Step(index, "final", reply, seconds=time.perf_counter() - start))
            return result
        obs = registry.call(action, data.get("action_input", {}))
        result.tool_errors += not obs.ok
        messages.append({"role": "user", "content": f"Observation: {obs.content}"})
        result.steps.append(Step(index, "action", reply, action, data.get("action_input"), obs.content, obs.ok,
                                 time.perf_counter() - start))
    return result


registry = ToolRegistry()
for fn in bookstore.TOOLS:
    registry.register(fn)
print([s["function"]["name"] for s in registry.specs()])
print(len(bookstore.TASKS), "task; ví dụ:", bookstore.TASKS[12])

In [ ]:
# explore: chạy 3 cách x 24 task (khoảng 10 tới 15 phút)
MODES = {
    "react": lambda client, q: run_react(client, q, registry, max_steps=8),
    "native": lambda client, q: run_native(client, q, registry, max_steps=8),
    "react_json": lambda client, q: run_react_json(client, q, registry, max_steps=8),
}
rows = []
for mode, agent in MODES.items():
    for task_id, group, question, expected in bookstore.TASKS:
        client = OllamaChat()
        start = time.perf_counter()
        res = agent(client, question)
        rows.append(dict(mode=mode, id=task_id, group=group, question=question, expected=expected,
                         ok=check_answer(res.answer, expected), stopped=res.stopped, answer=res.answer or "",
                         steps=[asdict(s) for s in res.steps], n_steps=len(res.steps), tool_errors=res.tool_errors,
                         format_errors=res.format_errors, seconds=time.perf_counter() - start,
                         prompt_tokens=max(client.prompt_tokens)))
        r = rows[-1]
        print(f"{mode:10s} {task_id} {'ĐÚNG' if r['ok'] else 'sai '} {r['stopped']:15s} {r['n_steps']} bước "
              f"{r['seconds']:5.1f}s | {r['answer'][:80]!r}")
df = pd.DataFrame(rows)

In [ ]:
# explore: bảng tổng hợp có khoảng tin cậy, so theo cặp
summary = []
for mode, g in df.groupby("mode", sort=False):
    k, n = int(g.ok.sum()), len(g)
    lo, hi = wilson_interval(k, n)
    summary.append(dict(mode=mode, dung=f"{k}/{n}", ci95=f"[{lo:.2f}, {hi:.2f}]", buoc_tb=round(g.n_steps.mean(), 2),
                        loi_tool=int(g.tool_errors.sum()), loi_dinh_dang=int(g.format_errors.sum()),
                        khong_tra_loi=int((g.stopped != "answer").sum()),
                        giay_trung_vi=round(g.seconds.median(), 1), token_prompt_tb=round(g.prompt_tokens.mean())))
print(pd.DataFrame(summary).to_string(index=False))
by_mode = {m: dict(zip(g.id, g.ok)) for m, g in df.groupby("mode", sort=False)}
for a, b in [("react", "native"), ("react", "react_json")]:
    only_a, only_b = discordant(by_mode[a], by_mode[b])
    print(f"{a} đúng mà {b} sai: {only_a} | {b} đúng mà {a} sai: {only_b}")
print(df.pivot_table(index="group", columns="mode", values="ok", aggfunc="sum", sort=False))

In [ ]:
# explore: phân loại lỗi và đọc vài trace
df["failure"] = [classify_failure(r) for r in rows]
print(pd.crosstab(df.failure, df["mode"]))
for mode in MODES:
    print(f"\n===== {mode}: các task sai =====")
    for r in df[(df["mode"] == mode) & ~df.ok].itertuples():
        trace = [(s["tool"], s["args"], (s["observation"] or "")[:60]) for s in r.steps if s["kind"] == "action"]
        print(f"{r.id} [{r.failure}] {r.question}\n   trả lời: {r.answer[:120]!r}\n   tool: {trace}")

In [ ]:
# explore: mở rộng (không chấm): thêm 20 tool gây nhiễu vào registry rồi chạy lại ReAct.
# Câu hỏi: nhiều tool trong context hơn thì agent chọn sai tool nhiều hơn không, và tốn thêm bao nhiêu token?
def make_distractor(name, doc):
    def tool(query: str) -> str:
        return "Dịch vụ tạm thời không khả dụng."
    tool.__name__, tool.__doc__ = name, doc + "\n\nArgs:\n    query: nội dung yêu cầu"
    return tool


DISTRACTORS = [("get_weather", "Thời tiết hiện tại của một thành phố."), ("send_email", "Gửi email."),
               ("create_calendar_event", "Tạo sự kiện trong lịch."), ("translate_text", "Dịch văn bản."),
               ("search_web", "Tìm kiếm trên internet."), ("get_stock_price", "Giá cổ phiếu."),
               ("book_flight", "Đặt vé máy bay."), ("get_news", "Tin tức mới nhất."),
               ("convert_units", "Đổi đơn vị đo lường."), ("get_traffic", "Tình hình giao thông."),
               ("set_reminder", "Đặt nhắc việc."), ("play_music", "Phát nhạc."),
               ("get_movie_showtimes", "Lịch chiếu phim."), ("order_food", "Đặt đồ ăn."),
               ("check_bank_balance", "Số dư tài khoản."), ("get_horoscope", "Tử vi hôm nay."),
               ("find_restaurant", "Tìm nhà hàng."), ("get_exchange_history", "Lịch sử tỷ giá theo ngày."),
               ("track_package", "Theo dõi bưu kiện."), ("get_library_hours", "Giờ mở cửa thư viện.")]
noisy = ToolRegistry()
for fn in bookstore.TOOLS:
    noisy.register(fn)
for name, doc in DISTRACTORS:
    noisy.register(make_distractor(name, doc))
noisy_rows = []
for task_id, group, question, expected in bookstore.TASKS:
    client = OllamaChat()
    res = run_react(client, question, noisy, max_steps=8)
    wrong_tool = sum(1 for s in res.steps if s.kind == "action" and s.tool in dict(DISTRACTORS))
    noisy_rows.append(dict(id=task_id, ok=check_answer(res.answer, expected), wrong_tool=wrong_tool,
                           prompt_tokens=max(client.prompt_tokens)))
noisy_df = pd.DataFrame(noisy_rows)
print("25 tool: đúng", int(noisy_df.ok.sum()), "/ 24 | gọi tool nhiễu:", int(noisy_df.wrong_tool.sum()),
      "| token prompt TB:", round(noisy_df.prompt_tokens.mean()))

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Khoảng tin cậy của ba cách chồng lên nhau tới đâu? Với 24 task, chênh 2 tới 3 task có nói được gì không? Nhìn số cặp
  lệch nhau của `discordant`, không chỉ hiệu hai tỉ lệ.
- Mỗi cách hỏng theo kiểu nào nhiều nhất (`classify_failure`)? Đọc trace: agent tự bịa giá hay đọc sai observation?
  Agent hỏi lại người dùng thay vì gọi tool tiếp là nhãn gì?
- Cách JSON chậm hơn ReAct văn bản bao nhiêu lần, và vì sao (đếm số bước, đọc output)?
- Đổi `seed` (hoặc `temperature` 0.7) rồi chạy lại một cách: tỉ lệ đúng dao động bao nhiêu? So với chênh lệch giữa các
  cách.

Sau đó làm phần trắc nghiệm của bài trong app.